# Task 3

1. Implement word embedding world:

-  either only one of the two: CBOW or Skip-gram

-  or use packages such as PyTorch, TensorFlow, or others to build a neural network and train the model,

- or implement embedding "look up" and print the most similar words for a given word from the vocabulary.

# Word2vec
- is an algorithm for representing words as vectors (or embeddings), which allows to model the meaning of words and their relationship with each other.

*Embeddings* - is the representation of objects (usually words or phrases) as vectors in higher dimension space. These vectors are numerical representations of objects, and their values often reflect semantic or syntactic properties of objects.

Example: suppose we want to find out how similar the words *"doctor"* and *"physician"* are. After learning the Word2Vec model for English texts, the vectors of these words will be close and the model can calculate similarities, for example, using the cosine distance between vectors.


**Note that for word2vec you can choose to implement only one of two models (CBOW or skip-gram)**

-  CBOW (Continuous Bag of Words)
Guess the current word from context.

- Skip-gram
Guess the contextual words of the current word.

CBOW and Skip-gram - neural network models - teach word vectors where the meaning of words "sew" in coordinates

In [1]:
## try different embedding sizes and windows and make conclusions

In [2]:
import os
import spacy
import torch
import numpy as np
from torch import nn
from torch.nn.functional import cosine_similarity
from collections import defaultdict
from tqdm import tqdm
import pandas as pd

In [3]:
base_path = os.getcwd()
file_path = os.path.abspath(os.path.join(base_path, "..", "data", "without_chap_and_title", "eng_Anne_full_abbr.txt"))
text = open(file_path, encoding="utf-8").read()

# tokenization
nlp = spacy.load("en_core_web_sm")

In [4]:
def tokenize_text(text):
    doc = nlp(text)
    return [[token.text.lower() for token in sent if not token.is_punct] for sent in doc.sents]

tokenized_sentences = tokenize_text(text)

In [5]:
# vocabulary
def build_vocab(tokenized):
    word_freq = defaultdict(int)
    for sent in tokenized:
        for token in sent:
            word_freq[token] += 1
    word2idx = {word: idx for idx, word in enumerate(word_freq)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

word2idx, idx2word = build_vocab(tokenized_sentences)
vocab_size = len(word2idx)

In [6]:
# training sample generation
def generate_cbow_data(tokenized, window_size=2):
    contexts = []
    targets = []
    for sentence in tokenized:
        if len(sentence) < 2 * window_size + 1:
            continue
        for i in range(window_size, len(sentence) - window_size):
            context = sentence[i - window_size:i] + sentence[i + 1:i + window_size + 1]
            target = sentence[i]
            try:
                contexts.append([word2idx[w] for w in context])
                targets.append(word2idx[target])
            except KeyError:
                continue
    return torch.tensor(contexts), torch.tensor(targets)

In [7]:
# CBOW model
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        mean_embeds = embeds.mean(dim=1)
        out = self.linear(mean_embeds)
        return out

In [8]:
# function to search for similar words
def most_similar(word, model, word2idx, idx2word, top_k=5):
    if word not in word2idx:
        return []
    word_idx = word2idx[word]
    word_vec = model.embeddings(torch.tensor([word_idx]))
    similarities = []
    for idx in range(vocab_size):
        if idx == word_idx:
            continue
        other_vec = model.embeddings(torch.tensor([idx]))
        sim = cosine_similarity(word_vec, other_vec).item()
        similarities.append((idx2word[idx], sim))
    return sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]

In [9]:
results = []

embedding_dims = [50, 100, 200]
window_sizes = [1, 2, 3]

In [10]:
for embedding_dim in embedding_dims:
    for window_size in window_sizes:
        print(f"\n Embedding dim = {embedding_dim}, Window size = {window_size}")

        # generate training data
        X, y = generate_cbow_data(tokenized_sentences, window_size=window_size)

        # init model
        model = CBOW(vocab_size, embedding_dim)
        loss_fn = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        # train (shortened for speed)
        for epoch in range(10):  # reduced epoch number
            outputs = model(X)
            loss = loss_fn(outputs, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # similar words
        sim_anne = most_similar("anne", model, word2idx, idx2word)
        sim_house = most_similar("house", model, word2idx, idx2word)

        # save results
        results.append({
            "embedding_dim": embedding_dim,
            "window_size": window_size,
            "final_loss": round(loss.item(), 4),
            "anne_sim": sim_anne,
            "house_sim": sim_house
        })

# output of results
df = pd.DataFrame(results)
print(df[["embedding_dim", "window_size", "final_loss"]])


 Embedding dim = 50, Window size = 1

 Embedding dim = 50, Window size = 2

 Embedding dim = 50, Window size = 3

 Embedding dim = 100, Window size = 1

 Embedding dim = 100, Window size = 2

 Embedding dim = 100, Window size = 3

 Embedding dim = 200, Window size = 1

 Embedding dim = 200, Window size = 2

 Embedding dim = 200, Window size = 3
   embedding_dim  window_size  final_loss
0             50            1      7.8668
1             50            2      8.0173
2             50            3      7.9541
3            100            1      7.0613
4            100            2      7.2133
5            100            3      7.1383
6            200            1      5.9136
7            200            2      6.1454
8            200            3      6.1099


In [11]:
# examples simialr words
for row in results:
    print(f"\nEmbedding dim = {row['embedding_dim']}, Window size = {row['window_size']}")
    print("Top similar to 'anne':", [w for w, _ in row['anne_sim']])
    print("Top similar to 'house':", [w for w, _ in row['house_sim']])


Embedding dim = 50, Window size = 1
Top similar to 'anne': ['wake', 'gestures', 'silky', 'blood', 'furiously']
Top similar to 'house': ['medallist', 'sunday', 'memories', 'remind', 'farmsteads']

Embedding dim = 50, Window size = 2
Top similar to 'anne': ['lithe', 'scotia', 'fancy', 'puzzled', 'obedience']
Top similar to 'house': ['prosaic', 'sarcastically', 'nearer', 'silk', 'watched']

Embedding dim = 50, Window size = 3
Top similar to 'anne': ['squadrons', 'winked', 'furniture', 'forgotten', 'echoing']
Top similar to 'house': ['bridal', 'wishes', 'minnie', 'third', 'puffs']

Embedding dim = 100, Window size = 1
Top similar to 'anne': ['with', 'command', 'answered', 'cardboard', 'higher']
Top similar to 'house': ['welled', '.on', 'continued', 'believed', 'rain']

Embedding dim = 100, Window size = 2
Top similar to 'anne': ['ways', ".won't", 'bend', 'unusually', 'rhyme']
Top similar to 'house': ['act', '.call', 'ungraciously', 'admired', 'cheery']

Embedding dim = 100, Window size = 